# 02_backtest.ipynb — 60/40 vs Permanent Portfolio vs Dual Momentum

**Propósito:** Comparación enfocada de tres estrategias benchmark/alternativas, independiente del
`backtest.ipynb` principal (que compara STRAT001/002/003). Consume:
- `daily_returns_STRAT002.csv` — 60/40 SPY/AGG
- `daily_returns_STRAT004.csv` — Permanent Portfolio (25/25/25/25 SPY/TLT/GLD/SHY)
- `daily_returns_STRAT005.csv` — Dual Momentum Cross-Sectional (Sharpe ranking)

**Pre-requisito:** `signals.ipynb` (o `signals_v2_dual_momentum.ipynb`) ya corrido — STRAT002,
STRAT004 y STRAT005 deben existir en `data/signals/`.
**Referencia:** MScFE SKILL §5, §5.2, López de Prado Ch.4


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

BASE    = Path('/content/drive/MyDrive/portfolio_quant')
CONFIG  = BASE / 'config'
DATA    = BASE / 'data'
SIGNALS = DATA / 'signals'

COMPARE_STRATS = ['STRAT002', 'STRAT004', 'STRAT005']
STRAT_LABELS = {
    'STRAT002': '60/40 SPY/AGG',
    'STRAT004': 'Permanent Portfolio',
    'STRAT005': 'Dual Momentum',
}

print('signals/ existe:', SIGNALS.exists())
for strat_id in COMPARE_STRATS:
    f = SIGNALS / f'daily_returns_{strat_id}.csv'
    print(f'  {strat_id}: {"OK" if f.exists() else "[FALTA] correr signals.ipynb"}')


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import norm
import warnings
warnings.filterwarnings('ignore')

TRADING_DAYS = 252

try:
    cr_df     = pd.read_csv(CONFIG / 'costs_rates.csv')
    rf_annual = float(cr_df.loc[cr_df['parameter'] == 'risk_free_rate', 'value'].iloc[0])
except Exception:
    rf_annual = 0.02
    print('[warn] rf_annual=0.02 por defecto')

print(f'rf_annual: {rf_annual:.2%} | TRADING_DAYS: {TRADING_DAYS}')


In [ ]:
# ── CARGAR LAS 3 SERIES A COMPARAR ────────────────────────────────────────────
all_returns = {}
for strat_id in COMPARE_STRATS:
    csv_path = SIGNALS / f'daily_returns_{strat_id}.csv'
    if not csv_path.exists():
        print(f'[SKIP] {strat_id}: archivo no encontrado, correr signals.ipynb primero')
        continue
    df = pd.read_csv(csv_path, parse_dates=['date'], index_col='date')
    label = STRAT_LABELS.get(strat_id, strat_id)
    all_returns[label] = df['daily_return'].dropna()
    print(f'  {label} ({strat_id}): {len(all_returns[label])} obs | '
          f'{all_returns[label].index.min().date()} → {all_returns[label].index.max().date()}')

if not all_returns:
    raise FileNotFoundError('Ninguna de las 3 estrategias está disponible. Correr signals.ipynb primero.')

print(f'\n{len(all_returns)}/3 estrategias cargadas.')


In [ ]:
# ── FUNCIONES DE MÉTRICAS ─────────────────────────────────────────────────────
# Idénticas a backtest.ipynb — Ref: MScFE SKILL §5.1 (performance_summary)

def compute_cagr(returns_daily: pd.Series, n_periods: int = TRADING_DAYS) -> float:
    """CAGR desde retornos diarios (compounding geométrico). MScFE SKILL §2.2."""
    n_years    = len(returns_daily) / n_periods
    cum_return = (1 + returns_daily).prod()
    return float(cum_return ** (1 / n_years) - 1)

def compute_annualized_vol(returns_daily: pd.Series, n_periods: int = TRADING_DAYS) -> float:
    """Vol anualizada: std_diaria × √252. MScFE SKILL §2.2."""
    return float(returns_daily.std() * np.sqrt(n_periods))

def compute_sharpe(returns_daily: pd.Series, rf: float = rf_annual,
                   n_periods: int = TRADING_DAYS) -> float:
    """
    Sharpe ratio anualizado. MScFE SKILL §5.1.
    EXAM INSIGHT: Annualize daily Sharpe by ×√252, never use daily Sharpe directly.
    """
    excess_daily = returns_daily - rf / n_periods
    if returns_daily.std() == 0:
        return np.nan
    return float(excess_daily.mean() / returns_daily.std(ddof=1) * np.sqrt(n_periods))

def compute_sortino(returns_daily: pd.Series, rf: float = rf_annual,
                    n_periods: int = TRADING_DAYS) -> float:
    """Sortino: usa solo retornos negativos en el denominador. MScFE SKILL §5.1."""
    ann_ret      = compute_cagr(returns_daily, n_periods)
    downside     = returns_daily[returns_daily < 0]
    downside_vol = downside.std() * np.sqrt(n_periods) if len(downside) > 0 else np.nan
    if not downside_vol or downside_vol == 0:
        return np.nan
    return float((ann_ret - rf) / downside_vol)

def compute_max_drawdown(returns_daily: pd.Series) -> float:
    """Max drawdown desde retornos simples. MScFE SKILL §5.1."""
    equity_curve = (1 + returns_daily).cumprod()
    running_max  = equity_curve.cummax()
    drawdown     = (equity_curve - running_max) / running_max
    return float(drawdown.min())

def compute_calmar(returns_daily: pd.Series, n_periods: int = TRADING_DAYS) -> float:
    """Calmar = CAGR / |MaxDD|."""
    cagr   = compute_cagr(returns_daily, n_periods)
    max_dd = compute_max_drawdown(returns_daily)
    return float(-cagr / max_dd) if max_dd < 0 else np.nan

def performance_summary(returns_daily: pd.Series, name: str = 'Strategy',
                        rf: float = rf_annual) -> pd.Series:
    """Resumen completo de performance. MScFE SKILL §5.1 pattern."""
    r = returns_daily.dropna()
    return pd.Series({
        'CAGR':    compute_cagr(r),
        'Vol':     compute_annualized_vol(r),
        'Sharpe':  compute_sharpe(r, rf),
        'Sortino': compute_sortino(r, rf),
        'Calmar':  compute_calmar(r),
        'MaxDD':   compute_max_drawdown(r),
        'Win%':    float((r > 0).mean()),
        'N_obs':   len(r),
    }, name=name)

def compute_var(returns_daily: pd.Series, alpha: float = 0.05) -> float:
    """VaR histórico al nivel alpha."""
    return float(np.percentile(returns_daily.dropna(), alpha * 100))

def compute_cvar(returns_daily: pd.Series, alpha: float = 0.05) -> float:
    """CVaR (Expected Shortfall)."""
    r = returns_daily.dropna()
    return float(r[r <= compute_var(r, alpha)].mean())

def risk_summary(returns_daily: pd.Series, name: str = 'Strategy') -> pd.Series:
    """VaR, CVaR, skewness, curtosis."""
    r = returns_daily.dropna()
    return pd.Series({
        'VaR_95':   compute_var(r, 0.05),
        'CVaR_95':  compute_cvar(r, 0.05),
        'VaR_99':   compute_var(r, 0.01),
        'CVaR_99':  compute_cvar(r, 0.01),
        'Skewness': float(r.skew()),
        'Kurtosis': float(r.kurtosis()),
    }, name=name)

print('Funciones definidas: performance_summary | risk_summary | compute_sharpe | compute_max_drawdown')


## Split IS / OOS

López de Prado: definir IS y OOS **antes** de mirar los resultados.  
Mismo criterio que `backtest.ipynb` (80/20). Editar si corresponde un split distinto.


In [ ]:
# ── DEFINIR SPLIT IS / OOS ────────────────────────────────────────────────────
all_start  = min(s.index.min() for s in all_returns.values())
all_end    = max(s.index.max() for s in all_returns.values())
total_days = (all_end - all_start).days

oos_start  = all_start + pd.Timedelta(days=int(total_days * 0.80))
IS_END     = oos_start - pd.Timedelta(days=1)
OOS_START  = oos_start

print(f'Período total: {all_start.date()} → {all_end.date()}')
print(f'IS:  {all_start.date()} → {IS_END.date()}')
print(f'OOS: {OOS_START.date()} → {all_end.date()}')

def slice_period(returns_dict, start, end):
    return {k: v.loc[start:end] for k, v in returns_dict.items()}

returns_is  = slice_period(all_returns, all_start, IS_END)
returns_oos = slice_period(all_returns, OOS_START, all_end)


In [ ]:
# ── TABLA DE PERFORMANCE IS / OOS ────────────────────────────────────────────
perf_is  = pd.DataFrame({k: performance_summary(v, k) for k, v in returns_is.items()}).T
perf_oos = pd.DataFrame({k: performance_summary(v, k) for k, v in returns_oos.items()}).T

def format_perf(df):
    d = df.copy()
    for col in ['CAGR', 'Vol', 'MaxDD', 'Win%']:
        if col in d.columns:
            d[col] = d[col].map('{:.2%}'.format)
    for col in ['Sharpe', 'Sortino', 'Calmar']:
        if col in d.columns:
            d[col] = d[col].map(lambda x: f'{x:.2f}' if pd.notna(x) else 'NaN')
    return d

print('=' * 60)
print('IN-SAMPLE (IS)')
print('=' * 60)
print(format_perf(perf_is).to_string())

print()
print('=' * 60)
print('OUT-OF-SAMPLE (OOS)')
print('=' * 60)
print(format_perf(perf_oos).to_string())


In [ ]:
# ── EQUITY CURVES + DRAWDOWN + ROLLING SHARPE ──────────────────────────────────
COLORS         = ['#1f77b4', '#2ca02c', '#9467bd']
WINDOW_ROLLING = 63

fig = plt.figure(figsize=(14, 11))
gs  = gridspec.GridSpec(3, 1, hspace=0.35)
ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1])
ax3 = fig.add_subplot(gs[2])

for (name, ret_series), color in zip(all_returns.items(), COLORS):
    r           = ret_series.dropna()
    equity      = (1 + r).cumprod()
    drawdown    = equity / equity.cummax() - 1
    roll_excess = r.rolling(WINDOW_ROLLING).mean() * TRADING_DAYS - rf_annual
    roll_vol    = r.rolling(WINDOW_ROLLING).std() * np.sqrt(TRADING_DAYS)
    roll_sharpe = roll_excess / roll_vol

    ax1.plot(equity,      label=name, color=color, linewidth=1.4)
    ax2.fill_between(drawdown.index, drawdown, 0, alpha=0.3, color=color)
    ax2.plot(drawdown,    color=color, linewidth=0.7)
    ax3.plot(roll_sharpe, color=color, linewidth=1.0, label=name)

for ax in [ax1, ax2, ax3]:
    ax.axvline(OOS_START, color='black', linewidth=1.2, linestyle='--', alpha=0.6, label='IS|OOS')

ax1.set_title('Equity curves (base 1) — 60/40 vs Permanent vs Dual Momentum', fontsize=11)
ax1.legend(fontsize=9); ax1.grid(alpha=0.3); ax1.set_ylabel('Valor')
ax2.set_title('Drawdown', fontsize=11)
ax2.grid(alpha=0.3); ax2.set_ylabel('DD')
ax3.set_title(f'Rolling Sharpe ({WINDOW_ROLLING}d)', fontsize=11)
ax3.axhline(0, color='k', linewidth=0.5)
ax3.legend(fontsize=9); ax3.grid(alpha=0.3); ax3.set_ylabel('Sharpe')

plt.savefig(DATA / 'equity_curves_02_backtest.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── RISK MEASURES ─────────────────────────────────────────────────────────────
risk_table = pd.DataFrame({k: risk_summary(v, k) for k, v in all_returns.items()}).T

def format_risk(df):
    d = df.copy()
    for col in ['VaR_95', 'CVaR_95', 'VaR_99', 'CVaR_99']:
        if col in d.columns:
            d[col] = d[col].map('{:.3%}'.format)
    for col in ['Skewness', 'Kurtosis']:
        if col in d.columns:
            d[col] = d[col].map('{:.3f}'.format)
    return d

print('=== RISK MEASURES (período completo) ===')
print(format_risk(risk_table).to_string())


In [ ]:
# ── DEFLATED SHARPE RATIO ────────────────────────────────────────────────────
# Ref: MScFE SKILL §5.2, López de Prado (2016)
# n_trials_total: contar 60/40, Permanent Portfolio y Dual Momentum como 3 trials
# independientes sobre estos datos IS. Incrementar si se re-testean variantes.

n_trials_total = 3

def deflated_sharpe_ratio(sharpe_observed: float, n_trials: int,
                           sharpe_std: float, n_obs: int,
                           skew: float = 0.0, kurt: float = 3.0) -> float:
    """
    DSR: probabilidad de que el Sharpe real sea > 0 ajustando por selección.
    Ref: MScFE SKILL §5.2, López de Prado (2016).
    """
    emc = 0.5772156649
    z1  = norm.ppf(1 - 1.0 / n_trials)
    z2  = norm.ppf(1 - 1.0 / (n_trials * np.e))
    expected_max_sr = sharpe_std * ((1 - emc) * z1 + emc * z2)
    numerator   = (sharpe_observed - expected_max_sr) * np.sqrt(n_obs - 1)
    denominator = np.sqrt(1 - skew * sharpe_observed + (kurt - 1) / 4.0 * sharpe_observed**2)
    if denominator == 0:
        return np.nan
    return float(norm.cdf(numerator / denominator))


sharpe_values_is = [compute_sharpe(v) for v in returns_is.values()]
sharpe_values_is = [s for s in sharpe_values_is if not np.isnan(s)]

if len(sharpe_values_is) >= 2:
    sharpe_best = max(sharpe_values_is)
    sharpe_std  = float(np.std(sharpe_values_is))
    n_obs_avg   = int(np.mean([len(v) for v in returns_is.values()]))
    dsr = deflated_sharpe_ratio(sharpe_best, n_trials_total, sharpe_std, n_obs_avg)
    print(f'=== DEFLATED SHARPE RATIO ===')
    print(f'Mejor Sharpe IS:    {sharpe_best:.3f}')
    print(f'Std Sharpe IS:      {sharpe_std:.3f}')
    print(f'n_trials_total:     {n_trials_total}')
    print(f'DSR (prob SR>0):    {dsr:.3f}')
    if dsr >= 0.95:
        print('✓ DSR ≥ 0.95 — confianza razonable')
    else:
        print('⚠ DSR < 0.95 — posible overfitting, aumentar OOS o reducir trials')
else:
    print('[INFO] Se necesitan ≥2 estrategias para calcular DSR.')


In [ ]:
# ── RESUMEN EJECUTIVO ─────────────────────────────────────────────────────────
print('=' * 65)
print('RESUMEN EJECUTIVO — 60/40 vs Permanent Portfolio vs Dual Momentum')
print(f'Período total:  {all_start.date()} → {all_end.date()}')
print(f'rf_annual:      {rf_annual:.2%} | TRADING_DAYS: {TRADING_DAYS}')
print('=' * 65)

for period_label, perf_df in [('IS', perf_is), ('OOS', perf_oos)]:
    print(f'\n--- {period_label} ---')
    print(format_perf(perf_df[['CAGR', 'Vol', 'Sharpe', 'Sortino', 'MaxDD', 'Win%']]).to_string())

print()
print('Registrar resultados en config/portfolio_strategies.md (STRAT002, STRAT004, STRAT005).')
